In [ ]:
# ── Dixon-Coles Poisson baseline ─────────────────────────────────────
import numpy as np
from scipy.stats import poisson
from scipy.optimize import minimize

# ── Logistic Regression multinomial ──────────────────────────────────
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import LabelEncoder

# ── Random Forest ─────────────────────────────────────────────────────
from sklearn.ensemble import RandomForestClassifier

# ── XGBoost softprob ─────────────────────────────────────────────────
from xgboost import XGBClassifier

# ── CatBoost ─────────────────────────────────────────────────────────
from catboost import CatBoostClassifier

# ── Calibration ───────────────────────────────────────────────────────
from sklearn.calibration import CalibratedClassifierCV  # isotonic

# ── Eval: Log Loss + Brier ────────────────────────────────────────────
from sklearn.metrics import log_loss, brier_score_loss
from sklearn.preprocessing import label_binarize

# ── RPS (no sklearn impl — define manually) ───────────────────────────
def rps(y_true_onehot: np.ndarray, y_prob: np.ndarray) -> float:
    """Ranked Probability Score for ordered 3-class WDL output."""
    cum_true = np.cumsum(y_true_onehot, axis=1)
    cum_pred = np.cumsum(y_prob, axis=1)
    return float(np.mean(np.sum((cum_pred - cum_true) ** 2, axis=1) / (y_true_onehot.shape[1] - 1)))

# ── Shared utils ──────────────────────────────────────────────────────
import pandas as pd
from sklearn.model_selection import TimeSeriesSplit
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler

# ── Optuna hyperparameter optimisation ───────────────────────────────
import optuna
from optuna.samplers import TPESampler
from optuna.pruners import MedianPruner

In [ ]:
def evaluate_model(
    name: str,
    y_true: np.ndarray,
    y_prob: np.ndarray,
    classes: list = ["H", "D", "A"],
) -> dict:
    """
    Evaluate a WDL probability model with RPS, Log Loss, and multiclass Brier Score.

    Args:
        name:    model label for display
        y_true:  1-D array of true labels e.g. ["H", "D", "A", "H", ...]
        y_prob:  (n_samples, 3) probability matrix — columns must match `classes` order
        classes: outcome order, default ["H", "D", "A"]

    Returns:
        dict of scores (lower = better for all three)
    """
    y_onehot = label_binarize(y_true, classes=classes)

    _rps   = rps(y_onehot, y_prob)
    _ll    = log_loss(y_true, y_prob, labels=classes)
    _brier = float(np.mean([
        brier_score_loss(y_onehot[:, i], y_prob[:, i])
        for i in range(len(classes))
    ]))

    results = {"model": name, "rps": _rps, "log_loss": _ll, "brier": _brier}

    print(f"\n{'─' * 40}")
    print(f"  {name}")
    print(f"{'─' * 40}")
    print(f"  RPS       {_rps:.4f}  (lower better)")
    print(f"  Log Loss  {_ll:.4f}  (lower better)")
    print(f"  Brier     {_brier:.4f}  (lower better)")

    return results
def model_predictions(
    model,
    X: np.ndarray | pd.DataFrame,
    y_true: np.ndarray | pd.Series,
    name: str,
    classes: list = ["H", "D", "A"],
) -> dict:
    """
    Automate predictions + evaluation for any sklearn-compatible model.

    Args:
        model:   fitted model with predict_proba()
        X:       feature matrix (test/holdout)
        y_true:  true outcome labels
        name:    model label for evaluate_model()
        classes: outcome order, default ["H", "D", "A"]

    Returns:
        dict with y_prob, y_pred, and all eval scores
    """
    y_prob = model.predict_proba(X)
    y_pred = model.predict(X)

    scores = evaluate_model(name, y_true, y_prob, classes=classes)

    return {
        **scores,
        "y_prob": y_prob,
        "y_pred": y_pred,
    }

def select_best_model(results: list[dict]) -> pd.DataFrame:
    """
    Rank models across RPS, Log Loss, and Brier via composite rank score.
    Lower rank sum = better overall model.
    """
    df = (
        pd.DataFrame(results)
        .drop(columns=["y_prob", "y_pred"])
        .set_index("model")
    )

    # rank each metric independently (1 = best)
    for col in ["rps", "log_loss", "brier"]:
        df[f"{col}_rank"] = df[col].rank()

    # composite = sum of ranks (lower = better overall)
    df["composite_rank"] = df[["rps_rank", "log_loss_rank", "brier_rank"]].sum(axis=1)

    best_name  = df.index[0]
    best_model = models[best_name]

    return best_name. best_model
